# Aula 4.3 — Desenhar bounding boxes com OpenCV

**Bloco 4 — OpenCV**  
**Projeto:** IC-FAPEMA — Detecção de capacetes em motociclistas via drone  
**Objetivo:** Desenhar retângulos e rótulos de detecção diretamente na imagem com OpenCV

---

## 📖 Tabela de Termos

<table style='border-collapse: collapse; width: 100%; font-size: 15px;'>
  <thead>
    <tr style='background-color: #2c2c2c; color: white;'>
      <th style='padding: 10px; text-align: left; border: 1px solid #555;'>Termo</th>
      <th style='padding: 10px; text-align: left; border: 1px solid #555;'>Conceito</th>
      <th style='padding: 10px; text-align: left; border: 1px solid #555; background-color: #e07b00;'>Função no código</th>
    </tr>
  </thead>
  <tbody>
    <tr style='background-color: #1e1e1e; color: #ddd;'>
      <td style='padding: 9px; border: 1px solid #444;'><b>rectangle()</b></td>
      <td style='padding: 9px; border: 1px solid #444;'>Desenha um retângulo na imagem</td>
      <td style='padding: 9px; border: 1px solid #444; color: #ffaa44;'><code>cv2.rectangle(img, pt1, pt2, cor, espessura)</code></td>
    </tr>
    <tr style='background-color: #262626; color: #ddd;'>
      <td style='padding: 9px; border: 1px solid #444;'><b>pt1 / pt2</b></td>
      <td style='padding: 9px; border: 1px solid #444;'><i>point 1 / point 2</i> — cantos opostos do retângulo (superior-esquerdo e inferior-direito)</td>
      <td style='padding: 9px; border: 1px solid #444; color: #ffaa44;'><code>pt1=(x1,y1)</code>, <code>pt2=(x2,y2)</code></td>
    </tr>
    <tr style='background-color: #1e1e1e; color: #ddd;'>
      <td style='padding: 9px; border: 1px solid #444;'><b>putText()</b></td>
      <td style='padding: 9px; border: 1px solid #444;'><i>colocar texto</i> — escreve texto sobre a imagem</td>
      <td style='padding: 9px; border: 1px solid #444; color: #ffaa44;'><code>cv2.putText(img, texto, pos, fonte, tam, cor, esp)</code></td>
    </tr>
    <tr style='background-color: #262626; color: #ddd;'>
      <td style='padding: 9px; border: 1px solid #444;'><b>FONT_HERSHEY_SIMPLEX</b></td>
      <td style='padding: 9px; border: 1px solid #444;'>Fonte padrão do OpenCV para texto</td>
      <td style='padding: 9px; border: 1px solid #444; color: #ffaa44;'><code>cv2.FONT_HERSHEY_SIMPLEX</code></td>
    </tr>
    <tr style='background-color: #1e1e1e; color: #ddd;'>
      <td style='padding: 9px; border: 1px solid #444;'><b>thickness</b> (espessura)</td>
      <td style='padding: 9px; border: 1px solid #444;'>Grossura da linha do retângulo, em pixels</td>
      <td style='padding: 9px; border: 1px solid #444; color: #ffaa44;'>parâmetro <code>thickness</code></td>
    </tr>
    <tr style='background-color: #262626; color: #ddd;'>
      <td style='padding: 9px; border: 1px solid #444;'><b>cor (tupla BGR)</b></td>
      <td style='padding: 9px; border: 1px solid #444;'>Cor passada como (B, G, R) — OpenCV é BGR, não RGB!</td>
      <td style='padding: 9px; border: 1px solid #444; color: #ffaa44;'><code>(0, 0, 255)</code> = vermelho</td>
    </tr>
    <tr style='background-color: #1e1e1e; color: #ddd;'>
      <td style='padding: 9px; border: 1px solid #444;'><b>in-place</b> (no lugar)</td>
      <td style='padding: 9px; border: 1px solid #444;'>O OpenCV desenha direto sobre o array da imagem — altera o original</td>
      <td style='padding: 9px; border: 1px solid #444; color: #ffaa44;'><code>cv2.rectangle()</code> não retorna nova imagem</td>
    </tr>
    <tr style='background-color: #262626; color: #ddd;'>
      <td style='padding: 9px; border: 1px solid #444;'><b>copy()</b></td>
      <td style='padding: 9px; border: 1px solid #444;'>Copia a imagem antes de desenhar, para preservar o original</td>
      <td style='padding: 9px; border: 1px solid #444; color: #ffaa44;'><code>imagem_copia = imagem.copy()</code></td>
    </tr>
  </tbody>
</table>

## 🧠 Conceito

Na Aula 3.1 você desenhou bounding boxes com **Matplotlib** (`patches.Rectangle`).  
Agora você vai fazer a mesma coisa com **OpenCV** (`cv2.rectangle`) — exatamente como o YOLO desenha as detecções internamente.

A diferença principal:

```
Matplotlib: desenha por CIMA da imagem exibida (não altera o array)
OpenCV:     desenha DENTRO do array da imagem (altera os pixels!)
```

Por isso, sempre que for desenhar com OpenCV, faça uma **cópia** da imagem primeiro:

```python
imagem_com_caixas = imagem.copy()        # preserva o original
cv2.rectangle(imagem_com_caixas, ...)    # desenha na cópia
```

**Outro ponto importante:** o `rectangle()` não usa (x_centro, y_centro, largura, altura) como o YOLO.  
Ele usa dois **pontos opostos** do retângulo:

```
pt1 = canto superior esquerdo (x1, y1)
pt2 = canto inferior direito  (x2, y2)
```

Então convertemos as coordenadas YOLO (normalizadas, centro) para esses dois pontos:

```
x1 = (x_centro - largura/2) × img_largura
y1 = (y_centro - altura/2)  × img_altura
x2 = (x_centro + largura/2) × img_largura
y2 = (y_centro + altura/2)  × img_altura
```

> No projeto real, é exatamente assim que o YOLO desenha as caixas de detecção sobre os frames do drone.

In [ ]:
# Importar bibliotecas
import cv2
import numpy as np
import matplotlib.pyplot as plt

## 💻 Código

In [ ]:
# ── 1. Criar uma imagem simulada do drone ─────────────────────────────────────
imagem = np.ones((480, 640, 3), dtype=np.uint8) * 200   # fundo cinza

In [ ]:
# ── 2. Detecções do YOLO (formato normalizado, igual à Aula 3.1) ─────────────
# classe 0 = com capacete | classe 1 = sem capacete
deteccoes = [
    {'classe': 1, 'x_centro': 0.52, 'y_centro': 0.38, 'largura': 0.20, 'altura': 0.15},
    {'classe': 0, 'x_centro': 0.75, 'y_centro': 0.60, 'largura': 0.18, 'altura': 0.20},
    {'classe': 1, 'x_centro': 0.30, 'y_centro': 0.45, 'largura': 0.22, 'altura': 0.17},
]

img_altura, img_largura = imagem.shape[:2]
print(f"Dimensões da imagem: {img_largura}x{img_altura}")

In [ ]:
# ── 3. Copiar a imagem antes de desenhar (preservar o original) ──────────────
imagem_com_caixas = imagem.copy()

In [ ]:
# ── 4. Desenhar cada bounding box ─────────────────────────────────────────────
for det in deteccoes:

    # Converter coordenadas YOLO → dois pontos (pt1, pt2) em pixels
    x1 = int((det['x_centro'] - det['largura'] / 2) * img_largura)
    y1 = int((det['y_centro'] - det['altura']  / 2) * img_altura)
    x2 = int((det['x_centro'] + det['largura'] / 2) * img_largura)
    y2 = int((det['y_centro'] + det['altura']  / 2) * img_altura)

    # Cor em BGR: vermelho = sem capacete, verde = com capacete
    cor   = (0, 0, 255) if det['classe'] == 1 else (0, 255, 0)
    label = 'sem capacete' if det['classe'] == 1 else 'com capacete'

    # Desenhar o retângulo da bounding box
    cv2.rectangle(imagem_com_caixas, (x1, y1), (x2, y2), cor, thickness=2)

    # Escrever o rótulo da classe acima da caixa
    cv2.putText(imagem_com_caixas, label, (x1, y1 - 8),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, cor, thickness=2)

print("Bounding boxes desenhadas!")

In [ ]:
# ── 5. Converter BGR → RGB para exibir corretamente ───────────────────────────
imagem_final = cv2.cvtColor(imagem_com_caixas, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(10, 6))
plt.imshow(imagem_final)
plt.title('Detecções desenhadas com OpenCV — rectangle() + putText()')
plt.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# ── 6. Confirmar que o original não foi alterado (por isso usamos .copy()) ───
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

ax1.imshow(cv2.cvtColor(imagem, cv2.COLOR_BGR2RGB))
ax1.set_title('Imagem ORIGINAL\n(sem alterações — graças ao .copy())')
ax1.axis('off')

ax2.imshow(imagem_final)
ax2.set_title('Imagem COM as bounding boxes')
ax2.axis('off')

plt.tight_layout()
plt.show()

---
## 📋 Resumo da Aula 4.3

<table style='border-collapse: collapse; width: 100%; font-size: 15px;'>
  <thead>
    <tr style='background-color: #2c2c2c; color: white;'>
      <th style='padding: 10px; border: 1px solid #555;'>O que aprendeu</th>
      <th style='padding: 10px; border: 1px solid #555; background-color: #e07b00;'>Função no código</th>
      <th style='padding: 10px; border: 1px solid #555;'>Para que serve no projeto</th>
    </tr>
  </thead>
  <tbody>
    <tr style='background-color: #1e1e1e; color: #ddd;'>
      <td style='padding: 9px; border: 1px solid #444;'>Converter centro → pontos</td>
      <td style='padding: 9px; border: 1px solid #444; color: #ffaa44;'><code>(x_centro ± largura/2) × img_largura</code></td>
      <td style='padding: 9px; border: 1px solid #444;'>Transformar coordenadas YOLO em pontos de desenho</td>
    </tr>
    <tr style='background-color: #262626; color: #ddd;'>
      <td style='padding: 9px; border: 1px solid #444;'>Desenhar retângulo</td>
      <td style='padding: 9px; border: 1px solid #444; color: #ffaa44;'><code>cv2.rectangle()</code></td>
      <td style='padding: 9px; border: 1px solid #444;'>Marcar a detecção sobre o frame</td>
    </tr>
    <tr style='background-color: #1e1e1e; color: #ddd;'>
      <td style='padding: 9px; border: 1px solid #444;'>Escrever rótulo</td>
      <td style='padding: 9px; border: 1px solid #444; color: #ffaa44;'><code>cv2.putText()</code></td>
      <td style='padding: 9px; border: 1px solid #444;'>Identificar a classe de cada detecção</td>
    </tr>
    <tr style='background-color: #262626; color: #ddd;'>
      <td style='padding: 9px; border: 1px solid #444;'>Preservar o original</td>
      <td style='padding: 9px; border: 1px solid #444; color: #ffaa44;'><code>imagem.copy()</code></td>
      <td style='padding: 9px; border: 1px solid #444;'>Não perder o frame bruto do drone</td>
    </tr>
  </tbody>
</table>